=============================================================
 Tata Gen AI Job Simulation — Customer Delinquency Predictor
 Covers: EDA → Cleaning → Feature Engineering → Logistic
         Regression → Evaluation → Fairness Analysis
=============================================================


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, brier_score_loss
)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

#### Generate synthetic dataset (mirrors the 500-record Geldium dataset from the task)

In [ ]:
np.random.seed(42)
N = 500

def generate_dataset(n=N):
    emp_status = np.random.choice(
        ["Employed", "employed", "EMP", "Self-employed", "Unemployed", "retired"],
        size=n, p=[0.35, 0.15, 0.10, 0.15, 0.15, 0.10]
    )
    age       = np.random.randint(18, 70, n)
    income    = np.where(
        np.isin(emp_status, ["Unemployed", "retired"]),
        np.random.normal(30_000, 10_000, n),
        np.random.normal(70_000, 25_000, n)
    ).clip(10_000, 200_000)
    credit_score      = np.random.normal(650, 80, n).clip(300, 850)
    credit_util       = np.random.beta(2, 5, n)
    credit_util       = np.where(np.random.random(n) < 0.03, credit_util * 1.5, credit_util)  # a few > 1.0
    missed_payments   = np.random.poisson(1.2, n).clip(0, 6)
    loan_balance      = np.random.normal(15_000, 8_000, n).clip(0, 50_000)
    dti               = (loan_balance / income.clip(1)).clip(0, 1)
    account_tenure    = np.random.randint(1, 120, n)

    # Payment history: Month_1 … Month_6
    statuses = ["On-Time", "Late", "Missed"]
    payment_cols = {
        f"Month_{i}": np.random.choice(statuses, n, p=[0.65, 0.20, 0.15])
        for i in range(1, 7)
    }

    location  = np.random.choice(["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"], n)
    card_type = np.random.choice(["Gold", "Platinum", "Standard"], n, p=[0.3, 0.2, 0.5])

    # Delinquency label — influenced by key risk factors
    risk_score = (
        (missed_payments >= 4).astype(int) * 2 +
        (credit_util > 0.8).astype(int) +
        (dti > 0.4).astype(int) +
        np.isin(emp_status, ["Unemployed"]).astype(int)
    )
    prob = 1 / (1 + np.exp(-(risk_score - 2.5)))
    delinquent = (np.random.random(n) < prob).astype(int)

    # Inject missing values
    income_idx = np.random.choice(n, 27, replace=False)
    loan_idx   = np.random.choice(n, 22, replace=False)
    cs_idx     = np.random.choice(n, 1,  replace=False)

    df = pd.DataFrame({
        "Age": age, "Income": income, "Employment_Status": emp_status,
        "Location": location, "Credit_Score": credit_score,
        "Credit_Utilization": credit_util, "Loan_Balance": loan_balance,
        "Debt_to_Income_Ratio": dti, "Missed_Payments": missed_payments,
        "Account_Tenure": account_tenure, "Credit_Card_Type": card_type,
        **payment_cols,
        "Delinquent_Account": delinquent
    })
    df.loc[income_idx, "Income"]       = np.nan
    df.loc[loan_idx,   "Loan_Balance"] = np.nan
    df.loc[cs_idx,     "Credit_Score"] = np.nan
    return df

df = generate_dataset()
print("=" * 55)
print("         TATA GEN AI — DELINQUENCY PREDICTOR")
print("=" * 55)
print(f"\nDataset shape : {df.shape}")
print(f"Delinquency rate: {df['Delinquent_Account'].mean():.1%}\n")

         TATA GEN AI — DELINQUENCY PREDICTOR

Dataset shape : (500, 18)
Delinquency rate: 10.8%



### TASK1 - EDA

In [ ]:
print("─" * 40)
print("TASK 1 · EDA")
print("─" * 40)

## 1a. Missing values
print("\n[Missing Values]")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 1b. Employment status inconsistency
print("\n[Employment_Status unique values]")
print(df["Employment_Status"].value_counts())

## 1c. Credit Utilization anomalies
print(f"\n[Credit_Utilization > 1.0]: {(df['Credit_Utilization'] > 1.0).sum()} rows")

## 1d. Key risk indicator distributions
print("\n[Delinquency rate by Missed_Payments]")
print(df.groupby("Missed_Payments")["Delinquent_Account"].mean().round(2))

print("\n[Delinquency rate by Employment_Status]")
print(df.groupby("Employment_Status")["Delinquent_Account"].mean().round(2).sort_values(ascending=False))

## 1e. Correlation quick look
num_cols = ["Age", "Income", "Credit_Score", "Credit_Utilization",
            "Missed_Payments", "Loan_Balance", "Debt_to_Income_Ratio",
            "Account_Tenure", "Delinquent_Account"]
print("\n[Correlation with Delinquent_Account]")
print(df[num_cols].corr()["Delinquent_Account"].sort_values(ascending=False).round(3))


────────────────────────────────────────
TASK 1 · EDA
────────────────────────────────────────

[Missing Values]
Income          27
Credit_Score     1
Loan_Balance    22
dtype: int64

[Employment_Status unique values]
Employment_Status
Employed         179
Self-employed     76
Unemployed        76
employed          62
retired           55
EMP               52
Name: count, dtype: int64

[Credit_Utilization > 1.0]: 0 rows

[Delinquency rate by Missed_Payments]
Missed_Payments
0    0.12
1    0.08
2    0.09
3    0.12
4    0.33
5    0.50
Name: Delinquent_Account, dtype: float64

[Delinquency rate by Employment_Status]
Employment_Status
Unemployed       0.26
Self-employed    0.11
retired          0.09
EMP              0.08
Employed         0.08
employed         0.05
Name: Delinquent_Account, dtype: float64

[Correlation with Delinquent_Account]
Delinquent_Account      1.000
Debt_to_Income_Ratio    0.159
Loan_Balance            0.098
Missed_Payments         0.071
Credit_Utilization      0.028

In [ ]:
print("\n" + "─" * 40)
print("TASK 2 · MODEL PIPELINE")
print("─" * 40)


────────────────────────────────────────
TASK 2 · MODEL PIPELINE
────────────────────────────────────────


### 2.a Data cleaning

In [ ]:
emp_map = {
    "employed": "Employed", "EMP": "Employed",
    "Self-employed": "Self-Employed",
    "retired": "Retired", "Unemployed": "Unemployed", "Employed": "Employed"
}
df["Employment_Status"] = df["Employment_Status"].map(emp_map).fillna("Other")

# Cap Credit_Utilization
df["Credit_Utilization_Capped"] = (df["Credit_Utilization"] > 1.0).astype(int)
df["Credit_Utilization"] = df["Credit_Utilization"].clip(upper=1.0)

# Impute Income: median by Employment_Status
df["Income_Imputed"] = df["Income"].isnull().astype(int)
df["Income"] = df.groupby("Employment_Status")["Income"].transform(
    lambda x: x.fillna(x.median())
)

# Impute Loan_Balance: median by Credit_Score bin
df["Loan_Balance_Imputed"] = df["Loan_Balance"].isnull().astype(int)
df["CS_bin"] = pd.cut(df["Credit_Score"].fillna(df["Credit_Score"].median()), bins=5, labels=False)
df["Loan_Balance"] = df.groupby("CS_bin")["Loan_Balance"].transform(
    lambda x: x.fillna(x.median())
)
df.drop(columns=["CS_bin"], inplace=True)

# Impute Credit_Score: mean
df["Credit_Score"] = df["Credit_Score"].fillna(df["Credit_Score"].mean())

print("\n[After cleaning — missing values]")
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() > 0 else "No missing values ✓")


[After cleaning — missing values]
No missing values ✓


### 2.b Feature enginnering

In [ ]:
payment_cols = [f"Month_{i}" for i in range(1, 7)]
for col in payment_cols:
    df[col] = df[col].str.strip()

df["Total_Late"]    = (df[payment_cols] == "Late").sum(axis=1)
df["Total_Missed"]  = (df[payment_cols] == "Missed").sum(axis=1)
df["Payment_Issues"]= df["Total_Late"] + df["Total_Missed"]
df["Recent_Trend"]  = (df[["Month_4", "Month_5", "Month_6"]]
                       .isin(["Late", "Missed"]).sum(axis=1))
df.drop(columns=payment_cols + ["Total_Late", "Total_Missed"], inplace=True)

print("\n[New features]: Payment_Issues, Recent_Trend, imputation flags")




[New features]: Payment_Issues, Recent_Trend, imputation flags


### 2.c Feature selection & encoding

In [ ]:
features_num = [
    "Age", "Income", "Credit_Score", "Credit_Utilization",
    "Missed_Payments", "Loan_Balance", "Debt_to_Income_Ratio",
    "Account_Tenure", "Payment_Issues", "Recent_Trend",
    "Income_Imputed", "Loan_Balance_Imputed", "Credit_Utilization_Capped"
]
features_cat = ["Employment_Status", "Credit_Card_Type", "Location"]
target       = "Delinquent_Account"

X = pd.get_dummies(df[features_num + features_cat], columns=features_cat, drop_first=True)
y = df[target]

# Scale numerical features
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X)
X_scaled  = pd.DataFrame(X_scaled, columns=X.columns)



### 2.d Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, stratify=y, random_state=42
)
print(f"\n[Split] Train: {len(X_train)} | Test: {len(X_test)}")



[Split] Train: 375 | Test: 125


### 2.e Logistic Regression

In [ ]:
model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred      = model.predict(X_test)
y_prob      = model.predict_proba(X_test)[:, 1]

print("\n" + "─" * 40)
print("MODEL EVALUATION")
print("─" * 40)
print(classification_report(y_test, y_pred, target_names=["Non-Delinquent", "Delinquent"]))

auc    = roc_auc_score(y_test, y_prob)
brier  = brier_score_loss(y_test, y_prob)
print(f"ROC-AUC    : {auc:.4f}  (target > 0.80)")
print(f"Brier Score: {brier:.4f} (target < 0.15)")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"  False Negatives (missed delinquents): {cm[1,0]}")
print(f"  False Positives (wrongly flagged):    {cm[0,1]}")


────────────────────────────────────────
MODEL EVALUATION
────────────────────────────────────────
                precision    recall  f1-score   support

Non-Delinquent       0.98      0.76      0.85       112
    Delinquent       0.29      0.85      0.43        13

      accuracy                           0.77       125
     macro avg       0.63      0.80      0.64       125
  weighted avg       0.91      0.77      0.81       125

ROC-AUC    : 0.7940  (target > 0.80)
Brier Score: 0.1839 (target < 0.15)

Confusion Matrix:
[[85 27]
 [ 2 11]]
  False Negatives (missed delinquents): 2
  False Positives (wrongly flagged):    27


### 2.f Threshold optimisation

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx   = f1_scores.argmax()
best_thr   = thresholds[best_idx]
print(f"\n[Optimal threshold]: {best_thr:.3f}  |  Best F1: {f1_scores[best_idx]:.4f}")

y_pred_opt = (y_prob >= best_thr).astype(int)
print("\n[With optimal threshold]")
print(classification_report(y_test, y_pred_opt, target_names=["Non-Delinquent", "Delinquent"]))


[Optimal threshold]: 0.608  |  Best F1: 0.4865

[With optimal threshold]
                precision    recall  f1-score   support

Non-Delinquent       0.96      0.87      0.91       112
    Delinquent       0.38      0.69      0.49        13

      accuracy                           0.85       125
     macro avg       0.67      0.78      0.70       125
  weighted avg       0.90      0.85      0.87       125



### 2.g Cross validation

In [ ]:
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_res = cross_validate(
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    X_scaled, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall"]
)
print("\n[5-Fold Cross-Validation]")
for metric, key in [("ROC-AUC","test_roc_auc"), ("F1","test_f1"),
                    ("Precision","test_precision"), ("Recall","test_recall")]:
    scores = cv_res[key]
    print(f"  {metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")


[5-Fold Cross-Validation]
  ROC-AUC   : 0.6555 ± 0.1025
  F1        : 0.2524 ± 0.0740
  Precision : 0.1693 ± 0.0471
  Recall    : 0.5036 ± 0.1811


### 2.h Fairness and Bias Assesment

In [ ]:
print("\n" + "─" * 40)
print("FAIRNESS ASSESSMENT")
print("─" * 40)

test_df = df.loc[X_test.index].copy()
test_df["y_pred"] = y_pred_opt
test_df["y_prob"] = y_prob

def fairness_report(group_col):
    grp = test_df.groupby(group_col).apply(
        lambda g: pd.Series({
            "Pred_Rate": g["y_pred"].mean(),
            "TPR":       g.loc[g["Delinquent_Account"]==1, "y_pred"].mean()
                         if g["Delinquent_Account"].sum() > 0 else np.nan,
            "Count":     len(g)
        })
    ).round(3)
    rates = grp["Pred_Rate"].dropna()
    di = rates.min() / rates.max() if rates.max() > 0 else np.nan
    print(f"\n  [{group_col}]  Disparate Impact Ratio: {di:.3f}  (target ≥ 0.80)")
    print(grp.to_string())

fairness_report("Employment_Status")
fairness_report("Location")
fairness_report("Credit_Card_Type")



────────────────────────────────────────
FAIRNESS ASSESSMENT
────────────────────────────────────────

  [Employment_Status]  Disparate Impact Ratio: 0.220  (target ≥ 0.80)
                   Pred_Rate    TPR  Count
Employment_Status                         
Employed               0.132  0.800   76.0
Retired                0.267  0.500   15.0
Self-Employed          0.167  0.333   24.0
Unemployed             0.600  1.000   10.0

  [Location]  Disparate Impact Ratio: 0.132  (target ≥ 0.80)
           Pred_Rate  TPR  Count
Location                        
Bangalore      0.524  1.0   21.0
Chennai        0.143  0.5   28.0
Delhi          0.190  0.5   21.0
Hyderabad      0.115  0.5   26.0
Mumbai         0.069  0.0   29.0

  [Credit_Card_Type]  Disparate Impact Ratio: 0.117  (target ≥ 0.80)
                  Pred_Rate    TPR  Count
Credit_Card_Type                         
Gold                  0.293  0.600   41.0
Platinum              0.409  0.800   22.0
Standard              0.048  0.667   

### 2.i Customer risk score

In [ ]:
print("\n" + "─" * 40)
print("SAMPLE CUSTOMER RISK SCORES")
print("─" * 40)

def risk_category(p):
    if p < 0.30: return "Low"
    elif p < 0.70: return "Medium"
    else: return "High"

sample = test_df.head(10)[["Age", "Income", "Missed_Payments",
                             "Credit_Utilization", "Delinquent_Account"]].copy()
sample["Risk_Prob%"] = (y_prob[:10] * 100).round(1)
sample["Risk_Level"] = sample["Risk_Prob%"].apply(lambda p: risk_category(p/100))
print(sample.to_string())


────────────────────────────────────────
SAMPLE CUSTOMER RISK SCORES
────────────────────────────────────────
     Age        Income  Missed_Payments  Credit_Utilization  Delinquent_Account  Risk_Prob% Risk_Level
300   43  43879.770149                2            0.422633                   0        42.3     Medium
204   62  39757.631429                1            0.156010                   0        59.4     Medium
176   62  53297.738659                3            0.304393                   0        41.4     Medium
147   21  85228.453023                1            0.078517                   0        26.0        Low
44    23  84868.858542                1            0.388800                   0        43.7     Medium
277   68  83416.318813                4            0.253897                   1        18.5        Low
126   37  20767.667539                1            0.747542                   1        63.5     Medium
1     26  24887.843236                0            0.302290      

### 2.j Feature Importance

In [ ]:
coefs = pd.Series(model.coef_[0], index=X.columns).abs().sort_values(ascending=False)
print("\n[Top 10 Most Important Features]")
print(coefs.head(10).round(4).to_string())


[Top 10 Most Important Features]
Location_Mumbai                 0.5501
Employment_Status_Unemployed    0.3830
Loan_Balance                    0.3776
Income                          0.3517
Location_Delhi                  0.3161
Credit_Card_Type_Standard       0.2798
Location_Chennai                0.2522
Credit_Card_Type_Platinum       0.2514
Credit_Utilization              0.2300
Missed_Payments                 0.1849


## VISUALISATIONS

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Tata Gen AI Simulation — Delinquency Prediction Dashboard", fontsize=14, fontweight="bold")

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0,0].plot(fpr, tpr, "b-", lw=2, label=f"AUC = {auc:.3f}")
axes[0,0].plot([0,1],[0,1],"k--")
axes[0,0].set(title="ROC Curve", xlabel="False Positive Rate", ylabel="True Positive Rate")
axes[0,0].legend()

# 2. Precision-Recall Curve
axes[0,1].plot(recalls, precisions, "g-", lw=2)
axes[0,1].axvline(recalls[best_idx], color="r", linestyle="--", label=f"Best Thr={best_thr:.2f}")
axes[0,1].set(title="Precision-Recall Curve", xlabel="Recall", ylabel="Precision")
axes[0,1].legend()

# 3. Confusion Matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0,2],
            xticklabels=["Non-Del","Del"], yticklabels=["Non-Del","Del"])
axes[0,2].set(title="Confusion Matrix", xlabel="Predicted", ylabel="Actual")

# 4. Delinquency rate by Missed_Payments
mp_rate = df.groupby("Missed_Payments")["Delinquent_Account"].mean()
axes[1,0].bar(mp_rate.index, mp_rate.values, color="coral")
axes[1,0].set(title="Delinquency Rate by Missed Payments",
              xlabel="Missed Payments", ylabel="Delinquency Rate")

# 5. Feature importance (top 10)
coefs.head(10).sort_values().plot(kind="barh", ax=axes[1,1], color="steelblue")
axes[1,1].set(title="Top 10 Feature Importances (|coef|)", xlabel="Coefficient Magnitude")

# 6. Risk score distribution
axes[1,2].hist(y_prob, bins=30, color="purple", alpha=0.7)
axes[1,2].axvline(best_thr, color="red", linestyle="--", label=f"Threshold={best_thr:.2f}")
axes[1,2].set(title="Predicted Risk Probability Distribution",
              xlabel="Delinquency Probability", ylabel="Count")
axes[1,2].legend()

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/tata_model_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n[Charts saved] → tata_model_dashboard.png")